# ED Triage AI — Comprehensive System Test

Tests the **full end-to-end pipeline** for 10 clinical scenarios:

```
patient dict
    -> edtriage-live endpoint  (probabilities + SHAP + safety_flag)
    -> retrieve_node           (top-5 RAG cases from Pinecone)
    -> analyze_node            (LLM reasons independently over model + SHAP + RAG)
    -> synthesize_node         (final_report)
```

**What to look for:**
- `Model` vs `Reconciled` — escalation means the agent overrode the raw model prediction
- `LLM ESI` — the LLM's independent recommendation before seeing the model's answer
- `Clinical Rationale` — did the LLM cite SHAP contradictions and RAG case patterns?
- `Flags` — RAG ESCALATION and LLM DISAGREEMENT are the two key escalation signals

**Run all cells top to bottom.**

In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'langgraph>=1.1.0', 'langchain-aws>=0.2.0', 'langchain-core>=0.3.0',
    'pinecone>=3.0.0', 'pandas>=2.2.0'], check=True)

import os, json, time
import boto3
import pandas as pd
from IPython.display import display

REPO_ROOT = os.path.expanduser('~/ed_triage_ai')
if os.path.join(REPO_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))

from agents.graph import triage_graph

ENDPOINT_NAME = 'edtriage-live'
_session    = boto3.Session(region_name=os.getenv('AWS_REGION', 'us-east-1'))
_sm_runtime = _session.client('sagemaker-runtime')
print('Setup complete.')

Setup complete.


In [3]:
def _build_triage_text(cc, hpi=''):
    cc  = ' '.join(str(cc  or '').split()[:24])
    hpi = ' '.join(str(hpi or '').split()[:160])
    parts = []
    if cc:  parts.append(f'Chief complaint: {cc}.')
    if cc:  parts.append(f'Presenting with {cc}.')
    if hpi: parts.append(f'History: {hpi}.')
    return ' '.join(parts)

def call_endpoint(patient):
    payload = {
        'triage_text':       _build_triage_text(patient.get('chief_complaint',''), patient.get('hpi','')),
        'heart_rate':        patient.get('heart_rate'),
        'sbp':               patient.get('systolic_bp'),
        'dbp':               patient.get('diastolic_bp'),
        'resp_rate':         patient.get('resp_rate'),
        'spo2':              patient.get('spo2'),
        'temp_f':            patient.get('temperature'),
        'age':               patient.get('age'),
        'arrival_transport': patient.get('arrival_transport', 'UNKNOWN'),
    }
    if patient.get('pain') is not None:
        payload['pain'] = patient['pain']
    resp = _sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME, ContentType='application/json',
        Accept='application/json', Body=json.dumps(payload))
    return json.loads(resp['Body'].read())

def run_pipeline(patient, thread_id):
    prediction = call_endpoint(patient)
    result = triage_graph.invoke(
        {'patient': patient, 'prediction': prediction},
        config={'configurable': {'thread_id': thread_id}})
    return result['final_report']

print('Helpers defined.')

Helpers defined.


In [4]:
TEST_CASES = [
    {'label':'1 · STEMI suspect','expected':'L1',
     'notes':'Strong text + severely abnormal vitals. Model and LLM should converge on L1 with no escalation needed.',
     'patient':{'age':68,'gender':'Male','arrival_transport':'AMBULANCE',
                'chief_complaint':'CHEST PAIN RADIATING LEFT ARM DIAPHORESIS',
                'hpi':'Crushing chest pain radiating to left arm with diaphoresis and shortness of breath. Patient diaphoretic, pale, and in acute distress.',
                'heart_rate':120,'systolic_bp':85,'diastolic_bp':55,'resp_rate':24,'spo2':91,'temperature':98.2,'pain':9}},
    {'label':'2 · Under-triage trap','expected':'L1/L2 - model may misfire L3',
     'notes':'Known failure mode: vague CC dilutes strong NEWS2 signal. SHAP should show contradictions. Agent should escalate.',
     'patient':{'age':55,'gender':'Male','arrival_transport':'AMBULANCE',
                'chief_complaint':'CHEST PAIN',
                'heart_rate':110,'systolic_bp':90,'diastolic_bp':60,'resp_rate':22,'spo2':94,'temperature':98.6,'pain':8}},
    {'label':'3 · Ankle sprain','expected':'L3',
     'notes':'Benign MSK. Model and LLM should agree on L3 with no flags.',
     'patient':{'age':28,'gender':'Male','arrival_transport':'WALK IN',
                'chief_complaint':'ANKLE PAIN',
                'hpi':'Twisted ankle playing basketball. Mild swelling, able to partially bear weight.',
                'heart_rate':72,'systolic_bp':120,'diastolic_bp':78,'resp_rate':14,'spo2':99,'temperature':98.4,'pain':4}},
    {'label':'4 · Appendicitis suspect','expected':'L2',
     'notes':'Focal RLQ pain + low-grade fever. Model may under-call. RAG should surface similar admitted cases.',
     'patient':{'age':35,'gender':'Female','arrival_transport':'UNKNOWN',
                'chief_complaint':'ABDOMINAL PAIN RIGHT LOWER QUADRANT',
                'hpi':'Worsening right lower quadrant pain over 12 hours with nausea, vomiting, and low-grade fever.',
                'heart_rate':98,'systolic_bp':115,'diastolic_bp':72,'resp_rate':18,'spo2':98,'temperature':100.8,'pain':7}},
    {'label':'5 · Sepsis suspect','expected':'L1',
     'notes':'Classic sepsis triad: fever + tachycardia + hypotension + confusion. Very high NEWS2.',
     'patient':{'age':72,'gender':'Female','arrival_transport':'AMBULANCE',
                'chief_complaint':'CONFUSION FEVER',
                'hpi':'Patient disoriented and febrile. History of recurrent UTIs. Possible urosepsis.',
                'heart_rate':118,'systolic_bp':88,'diastolic_bp':52,'resp_rate':26,'spo2':92,'temperature':103.1,'pain':5}},
    {'label':'6 · Acute stroke','expected':'L1',
     'notes':'Sudden focal neuro deficit, FAST positive. Strong text + transport should drive L1.',
     'patient':{'age':61,'gender':'Male','arrival_transport':'AMBULANCE',
                'chief_complaint':'SUDDEN WEAKNESS FACIAL DROOP',
                'hpi':'Sudden onset left-sided weakness and facial droop. Unable to speak clearly. Onset 40 minutes ago. FAST exam positive.',
                'heart_rate':88,'systolic_bp':178,'diastolic_bp':105,'resp_rate':16,'spo2':96,'temperature':98.4,'pain':2}},
    {'label':'7 · Pediatric fever','expected':'L2',
     'notes':'Age=3: HR=130 and RR=28 are normal for children but trigger adult NEWS2. Safety flag expected.',
     'patient':{'age':3,'gender':'Male','arrival_transport':'UNKNOWN',
                'chief_complaint':'FEVER IRRITABILITY',
                'hpi':'High fever and irritability in a 3-year-old. Not eating since yesterday. No rash or neck stiffness.',
                'heart_rate':130,'systolic_bp':95,'diastolic_bp':60,'resp_rate':28,'spo2':97,'temperature':103.2,'pain':6}},
    {'label':'8 · MVA poly-trauma','expected':'L1',
     'notes':'High-energy mechanism + hemodynamic instability. Text + vitals should converge on L1.',
     'patient':{'age':44,'gender':'Male','arrival_transport':'AMBULANCE',
                'chief_complaint':'MOTOR VEHICLE ACCIDENT CHEST ABDOMINAL PAIN',
                'hpi':'High-speed motor vehicle collision. Severe chest and abdominal pain. Seatbelt sign present. GCS 14.',
                'heart_rate':125,'systolic_bp':94,'diastolic_bp':64,'resp_rate':20,'spo2':96,'temperature':98.0,'pain':9}},
    {'label':'9 · Suicidal ideation','expected':'L2',
     'notes':'Normal vitals - psychiatric urgency invisible to vital-sign scores. LLM should recognise the text signal.',
     'patient':{'age':30,'gender':'Female','arrival_transport':'WALK IN',
                'chief_complaint':'SUICIDAL IDEATION',
                'hpi':'Active suicidal thoughts with hopelessness and severe depression. No specific plan but expressing intent.',
                'heart_rate':82,'systolic_bp':118,'diastolic_bp':75,'resp_rate':14,'spo2':99,'temperature':98.6,'pain':0}},
    {'label':'10 · Sparse text','expected':'Stress test',
     'notes':'Minimal CC - BERT near-noise. All signal from structured features. Tests fusion head and LLM reasoning under text poverty.',
     'patient':{'age':50,'gender':'Male','arrival_transport':'WALK IN',
                'chief_complaint':'PAIN',
                'heart_rate':104,'systolic_bp':135,'diastolic_bp':88,'resp_rate':19,'spo2':96,'temperature':99.1,'pain':5}},
]
print(f'Defined {len(TEST_CASES)} test scenarios.')

Defined 10 test scenarios.


In [5]:
# Each call: endpoint (SHAP) -> Pinecone RAG -> Bedrock LLM -> final_report
# Expect ~15-25s per scenario.
results = []
for i, tc in enumerate(TEST_CASES):
    t0 = time.time()
    try:
        report = run_pipeline(tc['patient'], thread_id=f'smoke-{i+1:02d}')
        elapsed = time.time() - t0
        escalated = report['reconciled_level'] != report['triage_level']
        results.append({'tc': tc, 'report': report, 'error': None})
        print(f"[{i+1:02d}] {tc['label']:<35}"
              f"  model={report['triage_level']:<25}"
              f"  llm=ESI-{report['llm_esi']}"
              f"  reconciled={report['reconciled_level']}"
              + (' ESCALATED' if escalated else '')
              + f'  ({elapsed:.0f}s)')
    except Exception as e:
        results.append({'tc': tc, 'report': None, 'error': str(e)})
        print(f"[{i+1:02d}] {tc['label']:<35}  ERROR: {e}")

print(f"\n{sum(1 for r in results if not r['error'])}/{len(TEST_CASES)} completed.")

[01] 1 · STEMI suspect                    model=L1-Critical                llm=ESI-1  reconciled=L1-Critical  (12s)
[02] 2 · Under-triage trap                model=L3-Urgent                  llm=ESI-2  reconciled=L2-Emergent ESCALATED  (10s)
[03] 3 · Ankle sprain                     model=L3-Urgent                  llm=ESI-3  reconciled=L3-Urgent  (7s)
[04] 4 · Appendicitis suspect             model=L3-Urgent                  llm=ESI-2  reconciled=L2-Emergent ESCALATED  (9s)
[05] 5 · Sepsis suspect                   model=L1-Critical                llm=ESI-1  reconciled=L1-Critical  (10s)
[06] 6 · Acute stroke                     model=L1-Critical                llm=ESI-1  reconciled=L1-Critical  (9s)
[07] 7 · Pediatric fever                  model=L3-Urgent                  llm=ESI-2  reconciled=L2-Emergent ESCALATED  (9s)
[08] 8 · MVA poly-trauma                  model=L1-Critical                llm=ESI-1  reconciled=L1-Critical  (10s)
[09] 9 · Suicidal ideation                model=

## Summary Table

| Column | Meaning |
|---|---|
| **Model** | Raw endpoint prediction |
| **LLM ESI** | LLM independent recommendation (before seeing the model) |
| **Reconciled** | Effective system recommendation — more cautious of model vs LLM |
| **Escalated** | Reconciled is more urgent than model prediction |
| **Safety** | Endpoint safety flag (NEWS2 conflict) |
| **Flags** | RAG+ = RAG majority more urgent | LLM+ = LLM disagrees |

Color by **Reconciled Level**: red=L1, orange=L2, yellow=L3

In [6]:
LEVEL_STYLE = {
    'L1-Critical':          'background-color:#ffcccc; font-weight:bold;',
    'L2-Emergent':          'background-color:#ffe0b2; font-weight:bold;',
    'L3-Urgent':            'background-color:#fff9c4;',
    'L3-Urgent/LessUrgent': 'background-color:#fff9c4;',
}

rows = []
for r in results:
    tc = r['tc']
    if r['error']:
        rows.append({'Scenario':tc['label'],'Expected':tc['expected'],'Model':'ERROR',
                     'Conf%':'','LLM ESI':'','Reconciled':'ERROR','Escalated':'',
                     'LLM Agrees':'','Safety':'','RAG Cases':'','Flags':r['error']})
        continue
    rpt = r['report']
    escalated = rpt['reconciled_level'] != rpt['triage_level']
    rows.append({
        'Scenario':   tc['label'],
        'Expected':   tc['expected'],
        'Model':      rpt['triage_level'],
        'Conf%':      f"{rpt['confidence_pct']}%",
        'LLM ESI':    f"ESI-{rpt['llm_esi']}" if rpt['llm_esi'] else '-',
        'Reconciled': rpt['reconciled_level'],
        'Escalated':  'YES' if escalated else '-',
        'LLM Agrees': 'YES' if rpt['llm_agreement'] else ('NO' if rpt['llm_agreement'] is False else '-'),
        'Safety':     'FLAG' if rpt.get('safety_flag') else '-',
        'RAG Cases':  rpt['cases_retrieved'],
        'Flags':      ' | '.join(
                          (['RAG+'] if any('RAG ESCALATION' in f for f in rpt['flags']) else []) +
                          (['LLM+'] if any('LLM DISAGREEMENT' in f for f in rpt['flags']) else [])
                      ) or '-',
    })

df = pd.DataFrame(rows)

def _style_row(row):
    base   = LEVEL_STYLE.get(row['Reconciled'], '')
    styles = [base] * len(row)
    cols   = df.columns.tolist()
    if row['Escalated'] == 'YES':
        styles[cols.index('Escalated')]  = 'background-color:#e53935; color:white; font-weight:bold;'
        styles[cols.index('Reconciled')] = 'background-color:#e53935; color:white; font-weight:bold;'
    if row['Safety'] == 'FLAG':
        styles[cols.index('Safety')] = 'background-color:#e53935; color:white; font-weight:bold;'
    if row['LLM Agrees'] == 'NO':
        styles[cols.index('LLM Agrees')] = 'background-color:#ef9a9a; font-weight:bold;'
    return styles

display(df.style.apply(_style_row, axis=1)
          .set_properties(**{'text-align':'left','white-space':'nowrap','padding':'4px 8px'})
          .set_table_styles([{'selector':'th','props':[('text-align','left'),('padding','4px 8px')]}]))

,Scenario,Expected,Model,Conf%,LLM ESI,Reconciled,Escalated,LLM Agrees,Safety,RAG Cases,Flags
0,1 · STEMI suspect,L1,L1-Critical,85%,ESI-1,L1-Critical,-,YES,-,5,-
1,2 · Under-triage trap,L1/L2 - model may misfire L3,L3-Urgent,39%,ESI-2,L2-Emergent,YES,NO,FLAG,5,RAG+ | LLM+
2,3 · Ankle sprain,L3,L3-Urgent,93%,ESI-3,L3-Urgent,-,YES,-,5,-
3,4 · Appendicitis suspect,L2,L3-Urgent,88%,ESI-2,L2-Emergent,YES,NO,-,5,LLM+
4,5 · Sepsis suspect,L1,L1-Critical,48%,ESI-1,L1-Critical,-,YES,-,5,-
5,6 · Acute stroke,L1,L1-Critical,75%,ESI-1,L1-Critical,-,YES,-,5,-
6,7 · Pediatric fever,L2,L3-Urgent,67%,ESI-2,L2-Emergent,YES,NO,FLAG,5,LLM+
7,8 · MVA poly-trauma,L1,L1-Critical,43%,ESI-1,L1-Critical,-,YES,-,5,-
8,9 · Suicidal ideation,L2,L3-Urgent,46%,ESI-2,L2-Emergent,YES,NO,-,5,RAG+ | LLM+
9,10 · Sparse text,Stress test,L3-Urgent,95%,ESI-3,L3-Urgent,-,YES,-,5,-


## Per-Scenario Deep Dive

Full agent output per scenario: model vs LLM vs reconciled, SHAP evidence, RAG cases, clinical rationale, and active flags.

In [7]:
for r in results:
    tc = r['tc']
    print('=' * 72)
    print(f"SCENARIO  : {tc['label']}")
    print(f"Expected  : {tc['expected']}")
    print(f"Notes     : {tc['notes']}")
    if r['error']:
        print(f"ERROR     : {r['error']}")
        continue

    rpt = r['report']
    p   = rpt['probabilities']
    escalated = rpt['reconciled_level'] != rpt['triage_level']

    print()
    print(f"  Model prediction : {rpt['triage_level']} ({rpt['confidence_pct']}% confidence)")
    print(f"  LLM independent  : ESI-{rpt['llm_esi']}  ({'AGREES' if rpt['llm_agreement'] else 'DISAGREES'} with model)")
    print(f"  Reconciled       : {rpt['reconciled_level']}" + (' -- ESCALATED' if escalated else ' (same as model)'))
    print(f"  Probabilities    : L1={p['L1-Critical']:.3f}  L2={p['L2-Emergent']:.3f}  L3={p['L3-Urgent']:.3f}")

    shap = rpt.get('shap_features') or []
    if shap:
        print()
        print('  SHAP Evidence:')
        for f in shap:
            tag = '[AGAINST]' if 'away' in f.get('direction','') else '[support]'
            print(f"    {f['feature']:<28} {f['shap']:+.4f}  {tag}  {f['direction']}")
    if rpt.get('safety_flag'):
        print(f"  SAFETY FLAG      : {rpt.get('safety_reason')}")

    print()
    print(f"  RAG Cases ({rpt['cases_retrieved']} retrieved, {rpt['retrieval_time_display']}):")
    for c in rpt['similar_cases']:
        print(f"    [{c['similarity']:.2f}] ESI-{c['triage_level']} | {c['chief_complaint']} -> {c['diagnosis']} | {c['outcome']}")

    print()
    print('  Clinical Rationale (LLM):')
    for line in rpt['clinical_rationale'].split('. '):
        if line.strip():
            print(f"    {line.strip()}.")

    active_flags = [f for f in rpt['flags'] if f.startswith('RAG ESCALATION') or f.startswith('LLM DISAGREEMENT')]
    if active_flags:
        print()
        print('  Flags:')
        for f in active_flags:
            print(f'    ! {f}')

print('=' * 72)

SCENARIO  : 1 · STEMI suspect
Expected  : L1
Notes     : Strong text + severely abnormal vitals. Model and LLM should converge on L1 with no escalation needed.

  Model prediction : L1-Critical (85% confidence)
  LLM independent  : ESI-1  (AGREES with model)
  Reconciled       : L1-Critical (same as model)
  Probabilities    : L1=0.855  L2=0.108  L3=0.037

  SHAP Evidence:
    sbp                          +1.3956  [support]  toward L1-Critical
    news2_score                  +0.8186  [support]  toward L1-Critical
    transport_ordinal            +0.1479  [support]  toward L1-Critical
    spo2                         +0.1333  [support]  toward L1-Critical
    mews_score                   +0.0899  [support]  toward L1-Critical

  RAG Cases (5 retrieved, 240 ms):
    [0.62] ESI-1 | L Arm pain -> CIRCULATORY DISEASE NEC | ADMITTED
    [0.57] ESI-3 | R Arm pain -> "Spinal stenosis | ADMITTED
    [0.56] ESI-3 | L Arm pain -> "CONGESTIVE HEART FAILURE | ADMITTED
    [0.55] ESI-3 | R Arm pain